In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS electronics_retailer_clg.gold_KPIs;

1. Monthly Revenue Trend
Merge Sales, Products, Exchange Rates. Calculate total revenue USD by month (2019).


In [0]:
%sql
-- Using sales_data_cube for Monthly Revenue Trend
SELECT 
  order_year AS year,
  order_month AS month,
  ROUND(SUM(revenue_usd), 2) AS revenue_usd
FROM electronics_retailer_clg.gold.sales_data_cube
WHERE order_year = 2021
GROUP BY order_year, order_month
ORDER BY month;

In [0]:
%sql
-- Store KPI 1: Monthly Revenue Trend using sales_data_cube
CREATE OR REPLACE TABLE electronics_retailer_clg.gold_KPIs.kpi_monthly_revenue AS
SELECT 
  order_year AS year,
  order_month AS month,
  ROUND(SUM(revenue_usd), 2) AS revenue_usd
FROM electronics_retailer_clg.gold.sales_data_cube
WHERE order_year = 2021

GROUP BY order_year, order_month
ORDER BY month;

2. Peak Month Analysis
From Q1, identify top 3 revenue months. Calculate % of annual total.


In [0]:
%sql
-- Preview the sales_data_cube
SELECT * FROM electronics_retailer_clg.gold.sales_data_cube
LIMIT 100;

In [0]:
%sql
-- Peak Months Analysis using sales_data_cube
WITH monthly AS (
  SELECT 
    order_month AS month,
    SUM(revenue_usd) AS revenue_usd
  FROM electronics_retailer_clg.gold.sales_data_cube
  WHERE order_year = 2019
  GROUP BY order_month
),
total AS (
  SELECT SUM(revenue_usd) AS total_revenue
  FROM electronics_retailer_clg.gold.sales_data_cube
  WHERE order_year = 2019
)

SELECT 
  m.month,
  ROUND(m.revenue_usd, 2) AS revenue_usd,
  ROUND((m.revenue_usd / t.total_revenue) * 100, 2) AS pct_of_total
FROM monthly m
CROSS JOIN total t
ORDER BY m.revenue_usd DESC
LIMIT 3;

In [0]:
%sql
-- Store KPI 2: Peak Month Analysis using sales_data_cube
CREATE OR REPLACE TABLE electronics_retailer_clg.gold_KPIs.kpi_peak_months AS
WITH monthly AS (
  SELECT 
    order_month AS month,
    SUM(revenue_usd) AS revenue_usd
  FROM electronics_retailer_clg.gold.sales_data_cube
  WHERE order_year = 2019
  GROUP BY order_month
),
total AS (
  SELECT SUM(revenue_usd) AS total_revenue
  FROM electronics_retailer_clg.gold.sales_data_cube
  WHERE order_year = 2019
)

SELECT 
  m.month,
  ROUND(m.revenue_usd, 2) AS revenue_usd,
  ROUND((m.revenue_usd / t.total_revenue) * 100, 2) AS pct_of_total
FROM monthly m
CROSS JOIN total t
ORDER BY m.revenue_usd DESC
LIMIT 3;

In [0]:
%sql
-- Store KPI 3: Holiday Drivers using sales_data_cube
CREATE OR REPLACE TABLE electronics_retailer_clg.gold_KPIs.kpi_holiday_drivers AS
WITH peak_months AS (
  SELECT 
    order_month AS month
  FROM electronics_retailer_clg.gold.sales_data_cube
  WHERE order_year = 2019
    AND order_quarter = 2
  GROUP BY order_month
  ORDER BY SUM(revenue_usd) DESC
  LIMIT 3
),

category_sales AS (
  SELECT 
    product_category,
    SUM(revenue_usd) AS revenue
  FROM electronics_retailer_clg.gold.sales_data_cube
  WHERE order_year = 2019
    AND order_month IN (SELECT month FROM peak_months)
  GROUP BY product_category
),

total AS (
  SELECT SUM(revenue) AS total_revenue FROM category_sales
)

SELECT 
  c.product_category,
  ROUND(c.revenue, 2) AS peak_month_revenue,
  ROUND((c.revenue / t.total_revenue) * 100, 2) AS pct_of_peak_total
FROM category_sales c
CROSS JOIN total t
ORDER BY peak_month_revenue DESC
LIMIT 3;

In [0]:
%sql
-- Store KPI 4: Delivery Performance using sales_data_cube
CREATE OR REPLACE TABLE electronics_retailer_clg.gold_KPIs.kpi_delivery_performance AS
SELECT 
  ROUND(AVG(delivery_time_days), 2) AS average_days,
  COUNT(*) AS total_orders
FROM electronics_retailer_clg.gold.sales_data_cube
WHERE delivery_date IS NOT NULL;

In [0]:
%sql
-- Store KPI 5: Country Delivery Issues using sales_data_cube
CREATE OR REPLACE TABLE electronics_retailer_clg.gold_KPIs.kpi_country_delivery AS
SELECT 
  store_country,
  ROUND(AVG(delivery_time_days), 2) AS avg_days,
  COUNT(*) AS order_count,
  ROUND(percentile_approx(delivery_time_days, 0.5), 2) AS median_days
FROM electronics_retailer_clg.gold.sales_data_cube
WHERE delivery_date IS NOT NULL
GROUP BY store_country
ORDER BY avg_days DESC
LIMIT 5;

3. Holiday Drivers
For Q2 peak months, show top 3 product categories driving revenue.

In [0]:
%sql
-- Holiday Drivers (Q2 Peak Months) using sales_data_cube

WITH peak_months AS (
  SELECT 
    order_month AS month
  FROM electronics_retailer_clg.gold.sales_data_cube
  WHERE order_year = 2019
    AND order_quarter = 2
  GROUP BY order_month
  ORDER BY SUM(revenue_usd) DESC
  LIMIT 3
),

category_sales AS (
  SELECT 
    product_category,
    SUM(revenue_usd) AS revenue
  FROM electronics_retailer_clg.gold.sales_data_cube
  WHERE order_year = 2019
    AND order_month IN (SELECT month FROM peak_months)
  GROUP BY product_category
),

total AS (
  SELECT SUM(revenue) AS total_revenue FROM category_sales
)

SELECT 
  c.product_category,
  ROUND(c.revenue, 2) AS peak_month_revenue,
  ROUND((c.revenue / t.total_revenue) * 100, 2) AS pct_of_peak_total
FROM category_sales c
CROSS JOIN total t
ORDER BY peak_month_revenue DESC
LIMIT 3;

4. Delivery Performance
Calculate overall avg delivery time across all orders.
Output: Average_Days | Total_Orders


In [0]:
%sql
-- Delivery Performance using sales_data_cube
SELECT 
  ROUND(AVG(delivery_time_days), 2) AS average_days,
  COUNT(*) AS total_orders
FROM electronics_retailer_clg.gold.sales_data_cube
WHERE delivery_date IS NOT NULL;

5. Country Delivery Issues
Avg delivery time by store country. Show slowest 5 countries.


In [0]:
%sql
-- Country Delivery Issues using sales_data_cube
SELECT 
  store_country,
  ROUND(AVG(delivery_time_days), 2) AS avg_days,
  COUNT(*) AS order_count,
  ROUND(percentile_approx(delivery_time_days, 0.5), 2) AS median_days
FROM electronics_retailer_clg.gold.sales_data_cube
WHERE delivery_date IS NOT NULL
GROUP BY store_country
ORDER BY avg_days DESC
LIMIT 5;

6. Channel Performance
AOV (revenue/orders) by Online vs In-Store across continents (Online = StoreKey.isna()).

In [0]:
%sql
-- Channel Performance: AOV by Online vs Store using sales_data_cube

WITH base AS (
  SELECT 
    customer_continent AS continent,
    order_number,
    revenue_usd,
    CASE 
      WHEN sales_channel = 'Online' THEN 'online'
      WHEN sales_channel = 'In-Store' THEN 'store'
      ELSE 'unknown'
    END AS channel
  FROM electronics_retailer_clg.gold.sales_data_cube
  WHERE customer_continent IS NOT NULL
),

agg AS (
  SELECT
    continent,
    channel,
    COUNT(DISTINCT order_number) AS orders,
    SUM(revenue_usd) AS revenue
  FROM base
  GROUP BY continent, channel
)

SELECT
  continent,

  -- AOV
  ROUND(
    SUM(CASE WHEN channel='online' THEN revenue END) /
    SUM(CASE WHEN channel='online' THEN orders END), 2
  ) AS aov_online,

  ROUND(
    SUM(CASE WHEN channel='store' THEN revenue END) /
    SUM(CASE WHEN channel='store' THEN orders END), 2
  ) AS aov_store,

  -- Order counts
  SUM(CASE WHEN channel='online' THEN orders END) AS online_orders,
  SUM(CASE WHEN channel='store' THEN orders END) AS store_orders

FROM agg
GROUP BY continent
ORDER BY continent;

In [0]:
%sql
-- Store KPI 6: Channel Performance using sales_data_cube
CREATE OR REPLACE TABLE electronics_retailer_clg.gold_KPIs.kpi_channel_performance AS
WITH base AS (
  SELECT 
    customer_continent AS continent,
    order_number,
    revenue_usd,
    CASE 
      WHEN sales_channel = 'Online' THEN 'online'
      WHEN sales_channel = 'In-Store' THEN 'store'
      ELSE 'unknown'
    END AS channel
  FROM electronics_retailer_clg.gold.sales_data_cube
  WHERE customer_continent IS NOT NULL
),

agg AS (
  SELECT
    continent,
    channel,
    COUNT(DISTINCT order_number) AS orders,
    SUM(revenue_usd) AS revenue
  FROM base
  GROUP BY continent, channel
)

SELECT
  continent,
  ROUND(
    SUM(CASE WHEN channel='online' THEN revenue END) /
    SUM(CASE WHEN channel='online' THEN orders END), 2
  ) AS aov_online,
  ROUND(
    SUM(CASE WHEN channel='store' THEN revenue END) /
    SUM(CASE WHEN channel='store' THEN orders END), 2
  ) AS aov_store,
  SUM(CASE WHEN channel='online' THEN orders END) AS online_orders,
  SUM(CASE WHEN channel='store' THEN orders END) AS store_orders
FROM agg
GROUP BY continent
ORDER BY continent;

In [0]:
%sql
-- Store KPI 7: Volume Leaders using sales_data_cube
CREATE OR REPLACE TABLE electronics_retailer_clg.gold_KPIs.kpi_volume_leaders AS
WITH total AS (
  SELECT SUM(units_sold) AS total_units 
  FROM electronics_retailer_clg.gold.sales_data_cube
)

SELECT 
  RANK() OVER (ORDER BY SUM(units_sold) DESC) AS rank,
  product_category,
  SUM(units_sold) AS units_sold,
  ROUND((SUM(units_sold)/t.total_units)*100, 2) AS pct_of_total_units
FROM electronics_retailer_clg.gold.sales_data_cube
CROSS JOIN total t
GROUP BY product_category, t.total_units
ORDER BY units_sold DESC
LIMIT 5;

In [0]:
%sql
-- Store KPI 8: Revenue Leaders using sales_data_cube
CREATE OR REPLACE TABLE electronics_retailer_clg.gold_KPIs.kpi_revenue_leaders AS
WITH total AS (
  SELECT SUM(revenue_usd) AS total_rev 
  FROM electronics_retailer_clg.gold.sales_data_cube
)

SELECT 
  RANK() OVER (ORDER BY SUM(revenue_usd) DESC) AS rank,
  product_category,
  ROUND(SUM(revenue_usd), 2) AS revenue_usd,
  ROUND((SUM(revenue_usd)/t.total_rev)*100, 2) AS pct_of_total_revenue
FROM electronics_retailer_clg.gold.sales_data_cube
CROSS JOIN total t
GROUP BY product_category, t.total_rev
ORDER BY revenue_usd DESC
LIMIT 5;

In [0]:
%sql
-- Store KPI 9: Customer Profile using sales_data_cube
CREATE OR REPLACE TABLE electronics_retailer_clg.gold_KPIs.kpi_customer_profile AS
SELECT 
  customer_continent AS continent,
  customer_gender AS gender,
  COUNT(DISTINCT customerkey) AS customer_count,
  ROUND(SUM(revenue_usd), 2) AS total_spend_usd,
  ROUND(SUM(revenue_usd)/COUNT(DISTINCT customerkey), 2) AS avg_spend_per_cust
FROM electronics_retailer_clg.gold.sales_data_cube
WHERE customerkey IS NOT NULL
  AND customer_continent IS NOT NULL
GROUP BY customer_continent, customer_gender;

In [0]:
%sql
-- Store KPI 10: Customer Loyalty using sales_data_cube
CREATE OR REPLACE TABLE electronics_retailer_clg.gold_KPIs.kpi_customer_loyalty AS
WITH cust_orders AS (
  SELECT 
    customerkey,
    customer_continent AS continent,
    COUNT(DISTINCT order_number) AS orders
  FROM electronics_retailer_clg.gold.sales_data_cube
  WHERE customerkey IS NOT NULL
    AND customer_continent IS NOT NULL
  GROUP BY customerkey, customer_continent
)

SELECT 
  continent,
  ROUND(
    (SUM(CASE WHEN orders>=2 THEN 1 ELSE 0 END) /
     COUNT(*)) * 100, 2
  ) AS repeat_rate_pct,
  COUNT(*) AS unique_customers,
  SUM(CASE WHEN orders>=2 THEN 1 ELSE 0 END) AS repeat_customers
FROM cust_orders
GROUP BY continent;

## KPI Tables in gold_KPIs Schema

All 10 KPIs have been materialized as permanent tables in `electronics_retailer_clg.gold_KPIs`:

**📊 Data Source:** All KPIs now use `electronics_retailer_clg.gold.sales_data_cube` - a comprehensive view that joins fact_sales with all dimension tables (dim_customers, dim_products, store) and includes pre-calculated time dimensions and delivery metrics.

| Table Name | KPI Description | Key Metrics |
|------------|----------------|-------------|
| **kpi_monthly_revenue** | Monthly Revenue Trend (2019) | year, month, revenue_usd |
| **kpi_peak_months** | Top 3 Revenue Months | month, revenue_usd, pct_of_total |
| **kpi_holiday_drivers** | Q2 Top Product Categories | product_category, peak_month_revenue, pct_of_peak_total |
| **kpi_delivery_performance** | Overall Delivery Metrics | average_days, total_orders |
| **kpi_country_delivery** | Slowest 5 Countries | store_country, avg_days, order_count, median_days |
| **kpi_channel_performance** | AOV by Channel & Continent | continent, aov_online, aov_store, online_orders, store_orders |
| **kpi_volume_leaders** | Top 5 by Units Sold | rank, product_category, units_sold, pct_of_total_units |
| **kpi_revenue_leaders** | Top 5 by Revenue | rank, product_category, revenue_usd, pct_of_total_revenue |
| **kpi_customer_profile** | Spending by Continent & Gender | continent, gender, customer_count, total_spend_usd, avg_spend_per_cust |
| **kpi_customer_loyalty** | Repeat Rate by Continent | continent, repeat_rate_pct, unique_customers, repeat_customers |

### Usage Examples:
```sql
-- Query any KPI table
SELECT * FROM electronics_retailer_clg.gold_KPIs.kpi_monthly_revenue;

-- List all KPI tables
SHOW TABLES IN electronics_retailer_clg.gold_KPIs;

-- Query the data cube directly
SELECT * FROM electronics_retailer_clg.gold.sales_data_cube LIMIT 100;
```

### Benefits of Using sales_data_cube:
- ✅ **Simplified Queries**: No need to join multiple tables
- ✅ **Pre-calculated Dimensions**: Time dimensions (year, quarter, month, day) ready to use
- ✅ **Enriched Customer Data**: Gender and continent from dim_customers
- ✅ **Product Details**: Category and pricing from dim_products
- ✅ **Store Information**: Country, state, size from store dimension
- ✅ **Delivery Metrics**: Pre-calculated delivery_time_days
- ✅ **Consistent Results**: Single source of truth for all KPIs

7. Volume Leaders
Top 5 product categories by total units sold.

In [0]:
%sql
-- Volume Leaders using sales_data_cube
WITH total AS (
  SELECT SUM(units_sold) AS total_units 
  FROM electronics_retailer_clg.gold.sales_data_cube
)

SELECT 
  RANK() OVER (ORDER BY SUM(units_sold) DESC) AS rank,
  product_category,
  SUM(units_sold) AS units_sold,
  ROUND((SUM(units_sold)/t.total_units)*100, 2) AS pct_of_total_units
FROM electronics_retailer_clg.gold.sales_data_cube
CROSS JOIN total t
GROUP BY product_category, t.total_units
ORDER BY units_sold DESC
LIMIT 5;


8. Revenue Leaders
Top 5 product categories by total revenue USD.

In [0]:
%sql
-- Revenue Leaders using sales_data_cube
WITH total AS (
  SELECT SUM(revenue_usd) AS total_rev 
  FROM electronics_retailer_clg.gold.sales_data_cube
)

SELECT 
  RANK() OVER (ORDER BY SUM(revenue_usd) DESC) AS rank,
  product_category,
  ROUND(SUM(revenue_usd), 2) AS revenue_usd,
  ROUND((SUM(revenue_usd)/t.total_rev)*100, 2) AS pct_of_total_revenue
FROM electronics_retailer_clg.gold.sales_data_cube
CROSS JOIN total t
GROUP BY product_category, t.total_rev
ORDER BY revenue_usd DESC
LIMIT 5;

9. Customer Profile
Customer count and spending by Continent × Gender.


In [0]:
%sql
-- Customer Profile using sales_data_cube
SELECT 
  customer_continent AS continent,
  customer_gender AS gender,
  COUNT(DISTINCT customerkey) AS customer_count,
  ROUND(SUM(revenue_usd), 2) AS total_spend_usd,
  ROUND(SUM(revenue_usd)/COUNT(DISTINCT customerkey), 2) AS avg_spend_per_cust
FROM electronics_retailer_clg.gold.sales_data_cube
WHERE customerkey IS NOT NULL
  AND customer_continent IS NOT NULL
GROUP BY customer_continent, customer_gender;

10. Customer Loyalty
Repeat customer rate (% with 2+ orders) by continent.


In [0]:
%sql
-- Customer Loyalty using sales_data_cube
WITH cust_orders AS (
  SELECT 
    customerkey,
    customer_continent AS continent,
    COUNT(DISTINCT order_number) AS orders
  FROM electronics_retailer_clg.gold.sales_data_cube
  WHERE customerkey IS NOT NULL
    AND customer_continent IS NOT NULL
  GROUP BY customerkey, customer_continent
)

SELECT 
  continent,
  ROUND(
    (SUM(CASE WHEN orders>=2 THEN 1 ELSE 0 END) /
     COUNT(*)) * 100, 2
  ) AS repeat_rate_pct,
  COUNT(*) AS unique_customers,
  SUM(CASE WHEN orders>=2 THEN 1 ELSE 0 END) AS repeat_customers
FROM cust_orders
GROUP BY continent;